# Kvasir-SEG: U-Net vs Attention U-Net vs UNet++ — folds 1–4

Settings before running (right-hand sidebar → *Session options*):
- **Accelerator: GPU T4 x2** (two folds train at once, one per GPU)
- **Internet: On** (installs albumentations 2.0.8 and downloads Kvasir-SEG)
- **Input:** the `unet-code` dataset made from `dist/unet-code.zip`

Then **Save Version → Save & Run All (Commit)** so it keeps running after you close the tab.
Kaggle stops a session after **12 hours**; the benchmark cell below estimates whether all 4 folds fit.

In [ ]:
# 1. Copy the code out of the read-only input folder
import glob, os, shutil
src = os.path.dirname(sorted(glob.glob('/kaggle/input/**/train.py', recursive=True))[0])
shutil.copytree(src, '/kaggle/working/UNet', dirs_exist_ok=True)
os.chdir('/kaggle/working/UNet')
print('code from', src, '->', os.getcwd())
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# 2. Install pinned albumentations, fetch + verify Kvasir-SEG (must match the fold-0 copy byte for byte)
!python kaggle/setup_kaggle.py

In [ ]:
# 3. Tests (models, metrics, split integrity, augmentation alignment); stop here if anything fails
!python -m pytest -q -x

In [ ]:
# 4. Timing: estimate hours for the planned folds before committing the session
FOLDS = [1, 2, 3, 4]
GPUS = [0, 1]
EPOCHS = 100
import re, subprocess, sys
epoch_seconds = {}
for model in ['unet', 'attention_unet', 'unetpp']:
    out = subprocess.run([sys.executable, 'train.py', '--model', model, '--benchmark', '15', '--workers', '2'],
                         capture_output=True, text=True).stdout
    print(out.strip().splitlines()[-1])
    epoch_seconds[model] = float(re.search(r'~(\d+)s of training', out).group(1)) * 1.15  # +15% for validation
fold_hours = sum(epoch_seconds.values()) * EPOCHS / 3600
folds_per_gpu = -(-len(FOLDS) // len(GPUS))
print(f'~{fold_hours:.1f} h per fold -> ~{fold_hours * folds_per_gpu:.1f} h with {len(GPUS)} GPUs in parallel '
      f'(both GPUs share 4 CPU cores, so allow some slack)')
if fold_hours * folds_per_gpu > 11:
    print('Too long for one 12 h session: set FOLDS = [1, 2] now and run [3, 4] in a second session, '
          'or add --amp to the training command below.')

In [ ]:
# 5. Train: each GPU works through its folds; logs in runs/kvasir/logs/<model>_fold<k>.log
folds = ' '.join(map(str, FOLDS)); gpus = ' '.join(map(str, GPUS))
!python scripts/run_folds.py --folds {folds} --gpus {gpus} -- --epochs {EPOCHS} --workers 2

In [ ]:
# 6. Results, then one zip to download (Output tab -> kvasir_runs.zip)
import json, pathlib
for p in sorted(pathlib.Path('runs/kvasir').glob('*/fold*/summary.json')):
    s = json.loads(p.read_text())
    print(f"{s['run']:28s} fold {s['fold']}  test Dice {s['test_dice']:.4f}  IoU {s['test_iou']:.4f}  "
          f"best epoch {s['best_epoch']:3d}  {s['train_minutes']:.0f} min")
!grep -l Traceback runs/kvasir/logs/*.err || echo 'no tracebacks'
!cd runs && zip -qr /kaggle/working/kvasir_runs.zip kvasir && ls -lh /kaggle/working/kvasir_runs.zip